In [1]:
import csv
from traceback import print_exc
import numpy as np
np.int=int
import pandas as pd
import os
import matplotlib.pyplot as plt


In [2]:
FILE_OUTPUT_DIR = 'all_spectra_dereddened_analysis'

<h1>
SNe True Ages

In [3]:
from utils.helper_functions import calculate_sn_ages

In [4]:
calculated_ages_data = calculate_sn_ages('cfasnIa_param.dat', 'cfasnIa_mjdspec.dat', os.path.join(FILE_OUTPUT_DIR,'cfa_supersnid_all_spectra_ages.csv'))

Processed 2603 spectra. Results saved to all_spectra_dereddened_analysis/cfa_supersnid_all_spectra_ages.csv


/Users/pxm588@student.bham.ac.uk/PhD/cfa_spectra_pipeline/utils/helper_functions.py:253: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  params = pd.read_csv(
/Users/pxm588@student.bham.ac.uk/PhD/cfa_spectra_pipeline/utils/helper_functions.py:265: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  spectra = pd.read_csv(


In [5]:
# print(calculated_ages_data.head())

<h1>
Adding SNe suptypes

In [6]:
from utils.helper_functions import find_sn_subtypes

In [7]:
subtype_data = find_sn_subtypes('cfasnIa_mjdspec.dat','branchwangclass.dat', 'wang')
# print(subtype_data.head())

Successfully matched 2603 spectra to wang subtypes.


/Users/pxm588@student.bham.ac.uk/PhD/cfa_spectra_pipeline/utils/helper_functions.py:305: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  subtypes = pd.read_csv(
/Users/pxm588@student.bham.ac.uk/PhD/cfa_spectra_pipeline/utils/helper_functions.py:317: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  spectra = pd.read_csv(


<h1>
SNR calculation

In [8]:
from utils.snr_finder_group import find_snr

In [9]:
snr_df = find_snr('all_spectra_dereddened', 'cfasnIa_param.dat')
# print(snr_df.head())

Running DER_SNR analysis...


In [10]:
# print(snr_df.head())

<h1>
Merging all the dataframes

In [11]:
file_df = pd.merge(snr_df, calculated_ages_data, on='Filename')
if 'SN_Name_y' in file_df.columns:
        file_df = file_df.rename(columns={'SN_Name_x': 'SN_Name'}).drop(columns=['SN_Name_y'])
# print(file_df.head())

spectra_properties_df = pd.merge(file_df, subtype_data, on='Filename', how='left')
if 'SN_Name_y' in spectra_properties_df.columns:
        spectra_properties_df = spectra_properties_df.rename(columns={'SN_Name_x': 'SN_Name'}).drop(columns=['SN_Name_y'])
# print(spectra_properties_df.head())


spectra_properties_df.to_csv('spectra_properties.csv', index=False)

In [12]:
def load_dm15_data(dat_file):
    dm15_map = {}
    with open(dat_file, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.split()
            if len(parts) >= 6:
                sn_name = parts[0].strip()
                try:
                    dm15 = float(parts[5])
                    # 9.99 often means no estimate in these datasets
                    if dm15 == 9.99:
                        dm15 = np.nan
                    dm15_map[sn_name] = dm15
                except ValueError:
                    continue
    return dm15_map

def update_csv(csv_file, dm15_map):
    df = pd.read_csv(csv_file)

    # Ensure SN_Name is string
    df['SN_Name'] = df['SN_Name'].astype(str)

    # Map dm15 values
    # Some SN names might have 'sn' prefix in the CSV but not in the DAT, or vice versa.
    # Let's handle both.
    def get_dm15(sn):
        if sn in dm15_map:
            return dm15_map[sn]
        if sn.startswith('sn') and sn[2:] in dm15_map:
            return dm15_map[sn[2:]]
        if f"sn{sn}" in dm15_map:
            return dm15_map[f"sn{sn}"]
        return np.nan

    df['dm15'] = df['SN_Name'].apply(get_dm15)

    # Count how many were found
    found_count = df['dm15'].notna().sum()
    print(f"Added dm15 values for {found_count} out of {len(df)} rows.")

    output_file = csv_file # Overwrite or new file? Usually user wants to update.
    df.to_csv(output_file, index=False)
    print(f"Updated {output_file}")


dat_path = 'cfasnIa_param.dat'
csv_path = 'spectra_properties.csv'

if os.path.exists(dat_path) and os.path.exists(csv_path):
    dm15_values = load_dm15_data(dat_path)
    update_csv(csv_path, dm15_values)
else:
    print("Missing required files.")


Added dm15 values for 2026 out of 2603 rows.
Updated spectra_properties.csv


<h1>
SNID calculations

In [13]:
from utils.helper_functions import get_output_files, parse_snid_file, calculate_simple_mean
from utils.helper_functions import calculate_bootstrap_median_topn


In [14]:
import pandas as pd
import os
import numpy as np

# --- 1. Define Paths ---
PROPERTIES_FILE = 'spectra_properties.csv'
SNID_OUTPUT_DIR = '/Users/pxm588@student.bham.ac.uk/Desktop/snid/cfa_pipeline/all_spectra_dereddened'
FINAL_OUTPUT_FILE = 'all_spectra_dereddened_analysis/all_spectra_found_dataset.csv'

# --- 2. Load the Existing Merged Data ---
# This contains Filename, SNR, SN_Name, True Age, Redshift, and Subtype
master_df = pd.read_csv(PROPERTIES_FILE)

# --- 3. Process SNID Outputs ---
snid_results = []
output_files = get_output_files(SNID_OUTPUT_DIR)

print(f"Processing {len(output_files)} SNID output files...")

for f in output_files:
    # Normalize filename to match the 'Filename' column in spectra_properties.csv
    # This matches your specific cleaning logic
    spectrum_filename = os.path.basename(f).replace('.output', '').replace('_snid', '.flm')

    # Run your specific SNID calculations
    ages, rlaps, redshifts = parse_snid_file(f)
    std_dev, bootstrap_age = calculate_bootstrap_median_topn(ages, rlaps, top_n=20)

    # Append results using the standard 'Filename' key for merging
    if bootstrap_age is not None:
        snid_results.append({
            'Filename': spectrum_filename,
            'bootstrap_age': round(bootstrap_age, 4),
            'snid_std_dev': round(std_dev, 4)
        })
    else:
        snid_results.append({
            'Filename': spectrum_filename,
            'bootstrap_age': np.nan,
            'snid_std_dev': np.nan
        })

# --- 4. Convert Results to DataFrame and Merge ---
df_snid = pd.DataFrame(snid_results)

# Perform a left merge: Keep all rows in master_df and add SNID data where it matches
final_df = pd.merge(master_df, df_snid, on='Filename', how='left')

# --- 5. Save and Report ---
final_df.to_csv(FINAL_OUTPUT_FILE, index=False)

print(f"--- Process Complete ---")
print(f"Final dataset with {len(final_df)} rows saved to: {FINAL_OUTPUT_FILE}")

# Check for missing SNID data
missing_count = final_df['bootstrap_age'].isna().sum()
if missing_count > 0:
    print(f"Note: {missing_count} spectra in your property list did not have a matching SNID output file.")

Processing 2585 SNID output files...
--- Process Complete ---
Final dataset with 2603 rows saved to: all_spectra_dereddened_analysis/all_spectra_found_dataset.csv
Note: 65 spectra in your property list did not have a matching SNID output file.


Files without .outputs from SNID are either too low S/N or couldn't be matched to type Ia templates and SNID prefers to match them to other types.

The additional missing bootstrap ages are due to the SNID outputs not containing enough matches.

